# DataSage Stage 3: Persona-Aware Answering GRPO Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/training/train_answering.ipynb)

Trains **Qwen2.5-3B** via [Unsloth](https://github.com/unslothai/unsloth) + [TRL GRPO](https://huggingface.co/docs/trl/grpo_trainer) to generate persona-aware, data-grounded answers.

The model learns to answer enterprise data questions tailored to 3 personas (Executive, Manager, IC) across 4 domains, with **Patronus Lynx** hallucination detection as a reward signal.

**Runtime:** T4 GPU (Colab free tier) | ~30-45 min

---

## 1. Setup

In [ ]:
# Verify GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "GPU not detected! Go to Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)")

In [ ]:
%%capture
!pip install unsloth vllm
!pip install trl>=0.26 transformers accelerate peft bitsandbytes
!pip install wandb datasets requests patronus

In [ ]:
# --- API Keys ---
# Option A: Use Colab Secrets (recommended) — add keys in the key icon on the left sidebar
# Option B: Paste them below
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    try:
        os.environ["PATRONUS_API_KEY"] = userdata.get("PATRONUS_API_KEY")
    except Exception:
        pass  # Optional
    print("Loaded keys from Colab Secrets")
except Exception:
    pass  # Fill manually below if not using Colab Secrets

# Uncomment and fill if not using Colab Secrets:
# os.environ["HF_TOKEN"] = "hf_..."
# os.environ["WANDB_API_KEY"] = "wandb_..."
# os.environ["PATRONUS_API_KEY"] = "sk-..."  # Optional: enables Patronus Lynx hallucination detection

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or above"
print(f"Patronus Lynx: {'ENABLED' if os.environ.get('PATRONUS_API_KEY') else 'DISABLED (using local fallback)'}")
print("Keys loaded.")

## 2. Load Model (Unsloth 4-bit)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {total:,} total, {trainable:,} trainable ({100*trainable/total:.1f}%)")

## 3. Dataset

64 answering prompts: 4 domains x 3 personas (Executive, Manager, Individual Contributor) x ~5 questions each.

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are a data-driven enterprise analyst. You answer questions about "
    "datasets across multiple domains (HR, Sales, Project Management, "
    "IT Operations) tailored to the audience persona.\n\n"
    "Personas:\n"
    "- Executive: Focus on costs, ROI, strategic risk, portfolio trends, "
    "year-over-year metrics. Use strategic-financial language.\n"
    "- Manager: Focus on team performance, operational health, process "
    "bottlenecks, capacity. Use operational-actionable language.\n"
    "- Individual Contributor: Focus on personal tasks, deadlines, what to "
    "do next, simple explanations. Use plain-personal language.\n\n"
    "Respond with a JSON answer:\n"
    '{"answer": "<your answer>", "cited_columns": ["col1", "col2"], '
    '"reasoning": "<chain-of-thought>"}\n\n'
    "Rules:\n"
    "1. Ground every claim in the data -- cite specific columns and statistics.\n"
    "2. Match your tone and vocabulary to the persona.\n"
    "3. Be concise but thorough -- Executives want bullet points, ICs want clear next steps.\n"
    "4. Never fabricate numbers -- if the data doesn't support a claim, say so.\n\n"
    "Think step by step: identify the persona, understand the question, "
    "examine the data, then answer."
)

TASK_PROMPTS = [
    # --- HR (16) ---
    "As an Executive: What is the overall attrition rate and its financial impact on the organization?",
    "As a Manager: Which departments have the highest turnover and what patterns do you see?",
    "As an Individual Contributor: Am I at risk of burnout based on the overtime data?",
    "As an Executive: How does our compensation compare to industry benchmarks across job roles?",
    "As a Manager: Which team members show the highest flight risk scores?",
    "As an Individual Contributor: What's the average tenure in my department and how do I compare?",
    "As an Executive: What's the year-over-year trend in employee satisfaction scores?",
    "As a Manager: How is overtime distributed across my team and is it sustainable?",
    "As an Individual Contributor: What factors most affect performance ratings?",
    "As an Executive: What's the ROI of reducing attrition by 5% based on salary data?",
    "As a Manager: Which roles are hardest to retain and what can we do about it?",
    "As an Individual Contributor: How does my job satisfaction compare to the department average?",
    "As an Executive: What are the top 3 strategic risks in our workforce data?",
    "As a Manager: What's the capacity utilization of the team based on current headcount?",
    "As an Individual Contributor: What development areas should I focus on based on performance data?",
    "As an Executive: Summarize the workforce health metrics for the quarterly board report.",
    # --- Sales (16) ---
    "As an Executive: What's the pipeline health and forecast accuracy for this quarter?",
    "As a Manager: Which deals are at risk of slipping from the forecast?",
    "As an Individual Contributor: What should I focus on to close my highest-probability deal?",
    "As an Executive: What's the revenue impact of deals stuck in Negotiation stage?",
    "As a Manager: How does our team's deal velocity compare across regions?",
    "As an Individual Contributor: Which of my deals has the best win probability?",
    "As an Executive: What's the conversion rate by stage and where are we losing deals?",
    "As a Manager: Which reps are below quota and what's their pipeline gap?",
    "As an Individual Contributor: What's the average time deals spend in my current stage?",
    "As an Executive: What's the competitive risk across our enterprise deals?",
    "As a Manager: How should I reallocate team resources based on pipeline data?",
    "As an Individual Contributor: What lead sources have the highest conversion for my product?",
    "As an Executive: Project the Q4 revenue based on current pipeline and win rates.",
    "As a Manager: What's the optimal deal mix by size category for our team?",
    "As an Individual Contributor: How can I improve my deal velocity based on the data?",
    "As an Executive: Summarize the sales pipeline metrics for the leadership review.",
    # --- Project Management (16) ---
    "As an Executive: Which projects are at highest risk of missing their deadlines?",
    "As a Manager: How is resource utilization distributed across the team?",
    "As an Individual Contributor: Which of my tasks should I prioritize based on dependencies?",
    "As an Executive: What's the portfolio-level on-time delivery rate?",
    "As a Manager: What's the burndown rate for the current sprint?",
    "As an Individual Contributor: Am I overallocated based on estimated vs actual hours?",
    "As an Executive: What's the financial impact of project delays in the portfolio?",
    "As a Manager: Which tasks are blocking the most downstream work?",
    "As an Individual Contributor: What's my completion rate compared to estimates?",
    "As an Executive: How does our delay probability trend compare to last quarter?",
    "As a Manager: Which team members need workload rebalancing?",
    "As an Individual Contributor: What are my upcoming deadlines and their risk levels?",
    "As an Executive: Summarize project health metrics for the steering committee.",
    "As a Manager: What process bottlenecks are causing the most delays?",
    "As an Individual Contributor: How many of my tasks have high dependency depth?",
    "As an Executive: What's the schedule risk across the critical path projects?",
    # --- IT Operations (16) ---
    "As an Executive: What's our SLA compliance rate and its trend this quarter?",
    "As a Manager: Which ticket categories have the worst resolution times?",
    "As an Individual Contributor: Which tickets assigned to me are closest to breaching SLA?",
    "As an Executive: What's the cost impact of SLA breaches this month?",
    "As a Manager: How should I prioritize the escalation queue?",
    "As an Individual Contributor: What's the resolution pattern for tickets like mine?",
    "As an Executive: What are the top 3 systemic issues driving incident volume?",
    "As a Manager: Which systems have the most recurring incidents?",
    "As an Individual Contributor: What's the typical escalation path for my ticket category?",
    "As an Executive: What's the mean time to resolution trend across priorities?",
    "As a Manager: How is the team's incident workload distributed?",
    "As an Individual Contributor: What resolution type is most common for my ticket type?",
    "As an Executive: Summarize the IT operations health for the monthly business review.",
    "As a Manager: Which areas need more staffing based on incident patterns?",
    "As an Individual Contributor: Are there recurring patterns in the incidents I'm handling?",
    "As an Executive: What's the customer impact distribution across open incidents?",
]

dataset = Dataset.from_dict({
    "prompt": [
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": p}]
        for p in TASK_PROMPTS
    ]
})

print(f"Dataset: {len(dataset)} prompts across 4 domains x 3 personas")

## 4. Reward Functions

Four reward signals:
1. **Environment reward** — steps through the deployed OpenEnv answering environment
2. **Patronus Lynx reward** — hallucination detection (with local faithfulness fallback)
3. **JSON format reward** — well-formed JSON with answer, cited columns, reasoning
4. **Persona reward** — persona-appropriate language and tone

In [ ]:
import json
import re
import requests
import time

ENV_URL = "https://ricalanis-datasage-answering.hf.space"

# --- Parser ---
def parse_answering_action(text):
    """Extract an answering action dict from model output."""
    m = re.search(r'\{[^{}]*"answer"[^{}]*\}', text, re.DOTALL)
    if m:
        try:
            data = json.loads(m.group())
            if "answer" in data:
                return data
        except json.JSONDecodeError:
            pass
    cited = re.findall(r'\b([A-Z][a-zA-Z]+(?:[A-Z][a-z]+)*)\b', text)
    return {"answer": text, "cited_columns": cited[:5], "reasoning": ""}


# --- Reward 1: Environment ---
_env_calls = 0
_env_total_reward = 0.0

def env_reward_fn(completions, **kwargs):
    """Step through the answering environment. Primary reward signal."""
    global _env_calls, _env_total_reward
    rewards = []
    for text in completions:
        try:
            action = parse_answering_action(text)
            requests.post(f"{ENV_URL}/reset", json={}, timeout=10)
            r = requests.post(f"{ENV_URL}/step", json={"action": action}, timeout=10)
            reward = float(r.json().get("reward", 0.0))
        except Exception:
            reward = 0.0
        rewards.append(reward)
    _env_calls += 1
    _env_total_reward += sum(rewards) / len(rewards)
    if _env_calls % 5 == 0:
        print(f"  [env_reward] call {_env_calls}, running avg: {_env_total_reward/_env_calls:.3f}")
    return rewards


# --- Reward 2: Patronus Lynx / local faithfulness ---
def local_faithfulness_fn(completions, **kwargs):
    """Local faithfulness heuristic (fallback for Patronus)."""
    rewards = []
    for text in completions:
        score = 0.0
        cited = re.findall(r'\b([A-Z][a-zA-Z]+(?:[A-Z][a-z]+)*)\b', text)
        if len(cited) >= 1: score += 0.3
        if len(cited) >= 3: score += 0.2
        if re.search(r'\d+\.?\d*%', text) or re.search(r'\b\d{2,}\b', text): score += 0.2
        if any(w in text.lower() for w in ["i believe", "probably", "might be", "i'm not sure"]): score -= 0.2
        if len(text.strip()) < 50: score -= 0.3
        rewards.append(max(0.0, min(1.0, score)))
    return rewards


def patronus_reward_fn(completions, **kwargs):
    """Use Patronus Lynx for hallucination detection. Falls back to local scorer."""
    if not os.environ.get("PATRONUS_API_KEY"):
        return local_faithfulness_fn(completions, **kwargs)
    try:
        import patronus
        patronus.init()
        from patronus import Patronus, RemoteEvaluator
        client = Patronus()
        lynx = RemoteEvaluator("lynx", "patronus:hallucination")
        context = kwargs.get("context", "")
        task_input = kwargs.get("task_input", "Answer the question based on the data.")
        rewards = []
        for text in completions:
            result = client.evaluate(evaluators=lynx, task_output=text, task_input=task_input, task_context=context)
            rewards.append(float(result.results[0].score))
        return rewards
    except Exception:
        return local_faithfulness_fn(completions, **kwargs)


# --- Reward 3: JSON format ---
def json_format_reward(completions, **kwargs):
    """Reward well-formed JSON answer actions."""
    rewards = []
    for text in completions:
        m = re.search(r'\{[^{}]*"answer"[^{}]*\}', text, re.DOTALL)
        if m:
            try:
                data = json.loads(m.group())
                has_answer = bool(data.get("answer", "").strip())
                has_cited = bool(data.get("cited_columns"))
                has_reasoning = bool(data.get("reasoning", "").strip())
                if has_answer and has_cited and has_reasoning: rewards.append(1.0)
                elif has_answer and has_cited: rewards.append(0.7)
                elif has_answer: rewards.append(0.4)
                else: rewards.append(0.2)
            except (json.JSONDecodeError, AttributeError):
                rewards.append(0.2)
        else:
            rewards.append(0.0)
    return rewards


# --- Reward 4: Persona alignment ---
def persona_reward(completions, **kwargs):
    """Reward persona-appropriate language."""
    exec_markers = ["roi", "revenue", "cost", "strategic", "quarter",
                    "year-over-year", "portfolio", "margin", "growth", "benchmark", "trend"]
    mgr_markers = ["team", "performance", "bottleneck", "capacity",
                   "action", "priority", "escalation", "delivery", "utilization"]
    ic_markers = ["my", "i should", "next step", "deadline", "help",
                  "understand", "focus on", "assigned to me"]
    rewards = []
    for text in completions:
        t = text.lower()
        hits = max(
            sum(1 for m in exec_markers if m in t),
            sum(1 for m in mgr_markers if m in t),
            sum(1 for m in ic_markers if m in t),
        )
        if hits >= 4: rewards.append(0.5)
        elif hits >= 2: rewards.append(0.3)
        elif hits >= 1: rewards.append(0.1)
        else: rewards.append(0.0)
    return rewards


# Quick env sanity check
try:
    r = requests.get(f"{ENV_URL}/health", timeout=10)
    print(f"Environment health: {r.status_code}")
except Exception as e:
    print(f"WARNING: Environment not reachable ({e}). Training will use fallback rewards.")

## 5. Train with GRPO

In [ ]:
from trl import GRPOConfig, GRPOTrainer

os.environ["WANDB_PROJECT"] = "datasage"

training_args = GRPOConfig(
    output_dir="./outputs/answering-grpo",
    learning_rate=3e-6,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=256,
    max_prompt_length=512,
    logging_steps=1,
    save_steps=50,
    bf16=True,
    use_vllm=True,
    vllm_mode="colocate",
    report_to="wandb",
    run_name="datasage-answering-grpo",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[
        env_reward_fn,         # Primary: environment reward
        patronus_reward_fn,    # Faithfulness: Patronus Lynx (with local fallback)
        json_format_reward,    # Auxiliary: structured output
        persona_reward,        # Auxiliary: persona alignment
    ],
)

print(f"Starting GRPO training...")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Generations per prompt: {training_args.num_generations}")
print(f"  Reward functions: env, patronus_lynx, json_format, persona")
print()

t_start = time.time()
trainer.train()
elapsed = (time.time() - t_start) / 60
print(f"\nTraining complete in {elapsed:.1f} minutes")
print(f"Environment reward calls: {_env_calls}, avg reward: {_env_total_reward/_env_calls:.3f}" if _env_calls else "")

## 6. Save & Push to Hub

In [ ]:
HF_MODEL_REPO = "ricalanis/datasage-qwen-answering"

trainer.save_model("./outputs/answering-grpo-final")
tokenizer.save_pretrained("./outputs/answering-grpo-final")
print("Model saved locally.")

trainer.push_to_hub(HF_MODEL_REPO)
print(f"Pushed to https://huggingface.co/{HF_MODEL_REPO}")